# Pelatihan BiLSTM + fastText ABSA SATUSEHAT

1. Unggah notebook ini lewat **File → Upload notebook** di Google Colab.
2. Pilih **Runtime → Change runtime type → T4 GPU**, lalu **Run all / Jalankan semua**.
3. Tunggu unduhan fastText dan pelatihan tiga seed selesai. Izinkan unduhan ZIP dari browser.
4. Simpan **hasil_bilstm_lengkap.zip** dan berikan ZIP tersebut untuk verifikasi serta pembaruan laporan.

Model B4: embedding fastText Indonesia **cc.id.300**, dibekukan; UNK dilatih;
BiLSTM 128 unit per arah, max-pooling, delapan kepala empat label, dropout 0,5,
Adam 1e-3, batch 32, panjang 128, maksimal 30 epoch, patience 3. Seed **42, 43, 44**.
Input: **400 latih, 100 validasi, 150 uji manual** yang sama dengan IndoBERT.
Kosakata hanya dari latih; epoch dipilih berdasarkan validasi. Label sentimen uji hanya digunakan saat evaluasi akhir.

Unduhan vektor lengkap berukuran besar (sekitar GB). Notebook menyimpan hanya vektor kata latih
untuk pelatihan; kata tanpa vektor text menjadi UNK, bukan vektor subword dari model bin.
Sumber dan lisensi: [fastText resmi](https://fasttext.cc/docs/en/crawl-vectors.html),
Grave et al. (2018), CC BY-SA 3.0. Hash unduhan dicatat; hash ini bukan checksum resmi penerbit.

Notebook telah diperiksa dan diuji dengan data sintetis; **eksperimen B4 lengkap belum dijalankan**.
ZIP kemajuan dibuat setelah setiap seed dan saat terjadi kegagalan. File `/content` bersifat sementara;
seed yang terputus di tengah epoch harus diulang. Jangan hapus runtime sebelum mengunduh hasil.
Jika GPU tidak tersedia, notebook berhenti. Anda dapat memilih `DEVICE = "cpu"` di sel pertama
untuk menjalankan BiLSTM di CPU secara eksplisit; waktunya bisa lebih lama dan perangkat dicatat.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

DEVICE = "cuda"  # ubah ke "cpu" hanya jika memang memilih pelatihan CPU
SOURCE_COMMIT = "a410e0981f77c51f8ed81e92d40b980b3cf155fc"
REPO_URL = "https://github.com/bambangh31/web-mining-absa-kelompok-2.git"
REPO = Path("/content/web-mining-absa-bilstm")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "Sumber lokal berubah; gunakan folder REPO baru."
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_COMMIT], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
sys.path.insert(0, str(REPO))
import torch
assert DEVICE in ("cpu", "cuda")
if DEVICE == "cuda":
    assert torch.cuda.is_available(), "GPU tidak tersedia. Pilih runtime GPU atau ubah DEVICE ke cpu secara eksplisit."
print("Sumber:", SOURCE_COMMIT, "| Perangkat:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU")

## 1. Periksa data dan catat versi eksperimen

In [ ]:
import csv
import json
import hashlib
import importlib.metadata
from datetime import datetime, timezone
from src.model_data import Dataset
from src.model_bilstm import build_vocab
from src.fasttext_vectors import sha256_file, extract_subset

SEEDS = [42, 43, 44]
MAX_EPOCHS = 30
OUT = Path("/content/hasil_bilstm")
OUT.mkdir(parents=True, exist_ok=True)
expected = {
    "data/processed/reviews_clean.csv": "8186e667f08515ca886f42da2fd5d848659f501b5551a6946401c6c68b43af00",
    "data/annotations/split.csv": "67036c5db3ceb2188e4c06c76df4eb1b48199aaa3802c1f323dc14e1b04254ea",
    "data/annotations/verified.csv": "bb3421985df973a57a05f636bc26e60dac68097bf0898a9dbe778ce983fa1df2",
    "data/annotations/final_v1/gold.csv": "8dcea7e41a58d3ac79baa019a5b17c25b173e3094bd0bc9d29a6d3e6a017bade",
}
assert {p: sha256_file(p) for p in expected} == expected, "Input berbeda dari eksperimen IndoBERT."
dataset = Dataset("data/processed/reviews_clean.csv", "data/annotations/split.csv", "data/annotations/verified.csv")
# Hanya ID diambil di sini; sentimen uji digunakan pada sel evaluasi akhir.
with open("data/annotations/final_v1/gold.csv", encoding="utf-8-sig", newline="") as f:
    test_ids = [r["reviewId"] for r in csv.DictReader(f)]
assert len(dataset.train_ids) == 400 and len(dataset.val_ids) == 100
assert len(test_ids) == len(set(test_ids)) == 150
assert all(dataset.split[r] == "gold_test" for r in test_ids)
assert not set(test_ids) & (set(dataset.train_ids) | set(dataset.val_ids))
assert not set(dataset.train_ids) & set(dataset.val_ids)
vocab = build_vocab(dataset.text(r) for r in dataset.train_ids)
vocab_hash = hashlib.sha256("\n".join(vocab).encode()).hexdigest()
manifest = {
    "model_code": "B4", "source_commit": SOURCE_COMMIT, "seeds": SEEDS,
    "max_epochs": MAX_EPOCHS, "patience": 3, "n_train": 400, "n_validation": 100, "n_test": 150,
    "vocab_source": "train_only", "vocab_sha256": vocab_hash, "sha256_sources": expected,
    "sha256_code": {str(p): sha256_file(p) for p in sorted(Path("src").glob("*.py"))},
    "device": DEVICE, "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "python": sys.version, "packages": {p: importlib.metadata.version(p) for p in ["torch", "numpy"]},
    "started_utc": datetime.now(timezone.utc).isoformat(),
}
manifest_path = OUT / "manifest.json"
if manifest_path.exists():
    old = json.loads(manifest_path.read_text())
    for k in ["model_code", "source_commit", "seeds", "max_epochs", "vocab_sha256", "sha256_sources", "sha256_code", "device", "packages"]:
        assert old[k] == manifest[k], f"Konfigurasi {k} berubah; pilih folder OUT baru."
else:
    manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Latih 400, validasi 100, uji 150; kata latih:", len(vocab)-2)

## 2. Unduh fastText dan ambil vektor kata latih

Sel ini dapat berjalan lama saat unduhan pertama. Kemajuan dicetak per 128 MiB dan
setiap 250.000 baris. Tidak perlu memasang Ollama untuk BiLSTM.

In [ ]:
import urllib.request

VECTOR_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.id.300.vec.gz"
CACHE = Path("/content/fasttext-bilstm")
CACHE.mkdir(parents=True, exist_ok=True)
archive = CACHE / "cc.id.300.vec.gz"
VECTORS = CACHE / "cc.id.300.train.vec"
vector_meta = OUT / "fasttext.json"
last_progress = [-1]
def progress(blocks, block_size, total):
    done = blocks * block_size
    bucket = done // (128 * 1024 * 1024)
    if bucket != last_progress[0]:
        last_progress[0] = bucket
        print(f"Unduhan fastText: {done / 1024**2:.0f} MiB / {total / 1024**2:.0f} MiB", flush=True)

if not archive.exists():
    temporary = archive.with_suffix(".download.part")
    urllib.request.urlretrieve(VECTOR_URL, temporary, reporthook=progress)
    temporary.replace(archive)

cached = json.loads(vector_meta.read_text()) if vector_meta.exists() else None
archive_hash = sha256_file(archive)
if cached:
    assert cached["url"] == VECTOR_URL and cached["vocab_sha256"] == vocab_hash
    assert cached["archive_sha256"] == archive_hash, "Arsip fastText berbeda dari run sebelumnya."
if not (cached and VECTORS.exists() and cached["subset_sha256"] == sha256_file(VECTORS)):
    details = extract_subset(archive, VECTORS, vocab[2:])
    details.update(url=VECTOR_URL, vocab_sha256=vocab_hash, source="fastText Common Crawl Indonesian text vectors", license="CC BY-SA 3.0")
    if cached:
        assert details["subset_sha256"] == cached["subset_sha256"], "Subset berubah."
    vector_meta.write_text(json.dumps(details, indent=2), encoding="utf-8")
else:
    details = cached
assert details["n_matched"] > 0 and details["dim"] == 300
print("Cakupan kata latih:", f"{details['coverage']:.1%}", "| subset:", VECTORS)

## 3. Latih seed 42, 43, 44

F1 **validasi** dicetak setiap epoch. Saat seed selesai, ZIP kemajuan otomatis ditawarkan.
Jika dijalankan ulang dalam runtime yang sama, hasil seed yang sudah lengkap diperiksa lalu dilewati.
Belum ada checkpoint di tengah epoch. File sementara hilang jika VM Colab dihapus.

In [ ]:
import shutil
from google.colab import files
from src.absa_schema import ASPECTS, LABELS

def export_progress(name="hasil_bilstm_kemajuan"):
    zip_path = shutil.make_archive(f"/content/{name}", "zip", OUT)
    print("ZIP:", zip_path, flush=True)
    files.download(zip_path)
    return zip_path

def completed(seed):
    p = OUT / "predictions" / f"B4_{seed}.csv"
    m = OUT / "metrics" / f"B4_{seed}.json"
    if not (p.exists() and m.exists()):
        return False
    saved = json.loads(m.read_text())
    with p.open(encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))
    c = saved["config"]
    return (saved["seed"] == seed and saved["model_code"] == "B4"
            and c["vocab_sha256"] == vocab_hash and c["vocab_source"] == "train_only"
            and c["max_epochs"] == MAX_EPOCHS and c["device"] == DEVICE
            and c["n_latih"] == 400 and c["n_validasi"] == 100 and c["n_uji"] == 150
            and c["catatan"] is None and c["cakupan_fasttext"] > 0
            and len(rows) == 150 and {r["reviewId"] for r in rows} == set(test_ids)
            and all(r["model_code"] == "B4" and int(r["seed"]) == seed
                    and all(r[a] in LABELS for a in ASPECTS) for r in rows))

for seed in SEEDS:
    if completed(seed):
        print("Lewati hasil lengkap seed", seed)
        continue
    try:
        subprocess.run([sys.executable, "-u", "src/model_bilstm.py", "--labels", "data/annotations/verified.csv",
                        "--vectors", str(VECTORS), "--predict-from", "data/annotations/final_v1/gold.csv",
                        "--seeds", str(seed), "--max-epochs", str(MAX_EPOCHS), "--device", DEVICE,
                        "--out", str(OUT)], check=True)
        assert completed(seed), "Berkas hasil seed belum lengkap."
    except Exception as exc:
        (OUT / "error.json").write_text(json.dumps({"seed": seed, "error": str(exc)}, indent=2))
        export_progress("hasil_bilstm_parsial")
        raise
    export_progress(f"hasil_bilstm_seed_{seed}")

## 4. Evaluasi akhir dan unduh ZIP lengkap

Label uji hanya dipakai setelah pelatihan semua seed selesai. Sel ini menghitung ulang
metrik tiga seed dan sepuluh run pembanding yang sudah tersedia, termasuk confusion matrix per aspek.
Skor utama BiLSTM adalah rata-rata dan simpangan baku sampel tiga seed.

In [ ]:
from src.evaluate import evaluate, load_labels, average_runs

assert all(completed(seed) for seed in SEEDS), "Tiga seed belum lengkap."
gold = load_labels("data/annotations/final_v1/gold.csv")
assert set(gold) == set(test_ids)
scores = [evaluate(gold, load_labels(OUT / "predictions" / f"B4_{s}.csv"), test_ids) for s in SEEDS]
summary = {"B4": average_runs(scores)}
(OUT / "rata_rata_seed.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
specs = ["B1-LR=results/predictions/B1-LR_42.csv", "B1-SVM=results/predictions/B1-SVM_42.csv",
         "B1-NB=results/predictions/B1-NB_42.csv", "B3-Gemini=data/annotations/llm_gemini-3.5-flash-lite.csv"]
specs += [f"{code}_{s}=results/predictions/{code}_{s}.csv" for code in ["B2", "B2-L"] for s in SEEDS]
specs += [f"B4_{s}={OUT / 'predictions' / f'B4_{s}.csv'}" for s in SEEDS]
command = [sys.executable, "src/evaluate.py", "--gold", "data/annotations/final_v1/gold.csv",
           "--ids-split", "gold_test", "--out", str(OUT / "evaluasi")]
for spec in specs:
    command.extend(["--pred", spec])
subprocess.run(command, check=True)
(OUT / "completion.json").write_text(json.dumps({"status": "completed", "seeds": SEEDS,
    "n_test": len(test_ids), "finished_utc": datetime.now(timezone.utc).isoformat()}, indent=2))
print("Rata-rata ± SD tiga seed:")
print(json.dumps(summary, indent=2))
export_progress("hasil_bilstm_lengkap")
print("Simpan hasil_bilstm_lengkap.zip untuk verifikasi dan pembaruan laporan.")

## Unduh ulang hasil yang sudah ada (opsional)

Sel berikut hanya menawarkan unduhan ZIP isi folder hasil saat ini. Gunakan jika unduhan otomatis
diblokir browser. Jika pelatihan/evaluasi belum selesai, isi ZIP masih parsial.

In [ ]:
export_progress("hasil_bilstm_saat_ini")